# Dense Vector Retrieval [Step 2 - Semantic Search with Embeddings]

> **MLCourse - Agentic AI - Hybrid Search**

This notebook demonstrates dense vector retrieval using embeddings and
cosine similarity. We use ChromaDB as our vector store and show how
semantic search finds documents by meaning rather than exact keywords.
We compare dense results with BM25 to highlight when each approach wins.

In [1]:
# Import all libraries needed for this notebook.
import os                              # Path handling
import re                              # Tokenization for comparison
import chromadb                        # Vector database
from chromadb.utils import embedding_functions  # Built-in embedders

In [2]:
# ## Part 1: Loading and Chunking the Corpus
# Dense retrieval works on fixed-size chunks with some overlap to
# preserve context across boundaries.

CORPUS_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Remove Gutenberg header/footer if present.
if "*** START OF" in raw_text:
    raw_text = raw_text.split("*** START OF", 1)[1]
if "*** END OF" in raw_text:
    raw_text = raw_text.split("*** END OF", 1)[0]

# Split into paragraphs, then merge small ones into chunks of ~500 chars.
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

chunks = []
current_chunk = ""
for para in paragraphs:
    if len(current_chunk) + len(para) < 500:
        current_chunk += (" " if current_chunk else "") + para
    else:
        if current_chunk:
            chunks.append(current_chunk)
        current_chunk = para
if current_chunk:
    chunks.append(current_chunk)

print(f"Created {len(chunks)} text chunks")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")
print(f"Example chunk preview: {chunks[0][:100]}...")

Created 339 text chunks
Average chunk length: 423 chars
Example chunk preview: THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll ...


In [3]:
# ## Part 2: Understanding Embeddings
# Dense retrieval converts text into fixed-size vectors (embeddings)
# using a neural network. Similar meanings produce nearby vectors.
#
# Embedding model: all-MiniLM-L6-v2 (fast, 384 dimensions)
# This is a sentence-transformer model trained on semantic similarity.

# Create a ChromaDB client with a persistent directory.
CHROMA_DIR = r"D:\projects\python\MLCourse\03_agentic_ai\20_hybrid_search\chroma_store"

# Clean up any previous data for a fresh start.
import shutil
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Use the default embedding function (all-MiniLM-L6-v2).
ef = embedding_functions.DefaultEmbeddingFunction()

# Create or get a collection.
collection = client.get_or_create_collection(
    name="alice_chunks",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

print(f"ChromaDB collection created: {collection.name}")

ChromaDB collection created: alice_chunks


In [4]:
# ## Part 3: Indexing Documents
# We add all chunks to the vector store. ChromaDB computes embeddings
# automatically when we add documents.

# Add documents in batches for efficiency.
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    ids = [f"chunk_{j}" for j in range(i, i + len(batch))]
    collection.add(documents=batch, ids=ids)

print(f"Indexed {collection.count()} chunks into ChromaDB")
print(f"Each chunk is now a 384-dimensional vector in the index")

Indexed 339 chunks into ChromaDB
Each chunk is now a 384-dimensional vector in the index


In [5]:
# ## Part 4: Querying with Cosine Similarity
# Cosine similarity measures the angle between two vectors.
# It ranges from -1 (opposite) to 1 (identical direction).
# For normalized embeddings, cosine similarity is the standard metric.
#
# similarity = (A . B) / (||A|| * ||B||)

# Query: find passages about the rabbit.
query = "a white rabbit with pink eyes"
results = collection.query(query_texts=[query], n_results=5)

print(f"Query: '{query}'")
print()
for i, (doc, dist) in enumerate(
    zip(results["documents"][0], results["distances"][0])
):
    cosine_sim = 1 - dist  # ChromaDB returns L2 distance, convert
    preview = doc[:80].replace("\n", " ")
    print(f"  #{i+1} (dist={dist:.4f}): {preview}...")

Query: 'a white rabbit with pink eyes'

  #1 (dist=0.4687): After a time she heard a little pattering of feet in the distance, and she hasti...
  #2 (dist=0.4947): So Alice began telling them her adventures from the time when she first saw the ...
  #3 (dist=0.5098): Alice was not a bit hurt, and she jumped up on to her feet in a moment: she look...
  #4 (dist=0.5142): “It proves nothing of the sort!” said Alice. “Why, you don’t even know what they...
  #5 (dist=0.5169): “It’s—it’s a very fine day!” said a timid voice at her side. She was walking by ...


In [6]:
# ## Part 5: Semantic Search vs Keyword Search
# The power of dense retrieval is that it understands meaning.
# "a creature that hops" should match "rabbit" even though they
# share no keywords.

# Semantic query: no mention of "rabbit" but same meaning.
semantic_query = "a small creature that hops and has long ears"
results_semantic = collection.query(query_texts=[semantic_query], n_results=3)

print(f"Semantic query: '{semantic_query}'")
print("(No keyword overlap with 'rabbit')")
print()
for i, doc in enumerate(results_semantic["documents"][0]):
    preview = doc[:80].replace("\n", " ")
    print(f"  #{i+1}: {preview}...")

Semantic query: 'a small creature that hops and has long ears'
(No keyword overlap with 'rabbit')

  #1: As there seemed to be no chance of getting her hands up to her head, she tried t...
  #2: The long grass rustled at her feet as the White Rabbit hurried by—the frightened...
  #3: Just then she heard something splashing about in the pool a little way off, and ...


In [7]:
# Now do the same query with BM25 (keyword-based) for comparison.

from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_chunks = [simple_tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

query_tokens = simple_tokenize(semantic_query)
bm25_scores = bm25.get_scores(query_tokens)
top_bm25 = bm25_scores.argsort()[::-1][:3]

print("BM25 results for the same semantic query:")
for rank, idx in enumerate(top_bm25, 1):
    score = bm25_scores[idx]
    preview = chunks[idx][:80].replace("\n", " ")
    print(f"  #{rank} (score={score:.4f}): {preview}...")

print()
print("Dense retrieval found rabbit-related passages by meaning.")
print("BM25 returned less relevant results because no keywords matched.")

BM25 results for the same semantic query:
  #1 (score=10.2669): Alice was not a bit hurt, and she jumped up on to her feet in a moment: she look...
  #2 (score=10.1134): This question the Dodo could not answer without a great deal of thought, and it ...
  #3 (score=9.6948): Alice caught the baby with some difficulty, as it was a queer-shaped little crea...

Dense retrieval found rabbit-related passages by meaning.
BM25 returned less relevant results because no keywords matched.


In [8]:
# ## Part 6: When Dense Retrieval Works Best
# Dense retrieval excels when:
#
# 1. **Semantic matching** -- "animals that hop" finds "rabbit".
# 2. **Synonym handling** -- "happy" matches "joyful".
# 3. **Paraphrase queries** -- different words, same meaning.
# 4. **Long natural language queries** -- full sentences work well.
#
# Dense retrieval struggles with:
# 1. **Exact keywords** -- product codes, IDs, proper nouns.
# 2. **Rare domain terms** -- embeddings may not encode them well.
# 3. **Computational cost** -- requires GPU or significant CPU.
# 4. **Indexing time** -- embedding every document takes time.

print("When dense retrieval works best:")
print("  + Semantic matching and synonyms")
print("  + Paraphrase and natural language queries")
print("  + Cross-domain or cross-style matching")
print()
print("When dense retrieval struggles:")
print("  - Exact keyword and ID lookups")
print("  - Rare domain-specific terminology")
print("  - Higher computational cost than BM25")
print("  - Requires embedding model for every new document")

When dense retrieval works best:
  + Semantic matching and synonyms
  + Paraphrase and natural language queries
  + Cross-domain or cross-style matching

When dense retrieval struggles:
  - Exact keyword and ID lookups
  - Rare domain-specific terminology
  - Higher computational cost than BM25
  - Requires embedding model for every new document


In [9]:
# ## Part 7: Cosine Similarity Visualization
# Let us visualize how different queries relate to a passage.

import math

# Manual cosine similarity for demonstration.
def cosine_sim_manual(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# Get embeddings for a reference document and several queries.
ref_doc = chunks[0]  # Alice sitting by the river
queries = [
    "Alice sitting by the river",
    "a girl reading a book",
    "a white rabbit",
    "the queen of hearts",
]

# ChromaDB returns embeddings via the embedding function.
ref_embedding = ef([ref_doc])[0]
query_embeddings = ef(queries)

print("Cosine similarity of queries to first paragraph:")
print(f"Reference: '{ref_doc[:60]}...'")
print()
for q, qe in zip(queries, query_embeddings):
    sim = cosine_sim_manual(ref_embedding, qe)
    bar = "#" * int(sim * 40)
    print(f"  '{q}': {sim:.4f}  {bar}")

Cosine similarity of queries to first paragraph:
Reference: 'THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Ad...'

  'Alice sitting by the river': 0.3561  ##############
  'a girl reading a book': 0.3281  #############
  'a white rabbit': 0.2562  ##########
  'the queen of hearts': 0.2538  ##########


In [10]:
# ## Part 8: Key Takeaways
# Dense retrieval adds a semantic layer that keyword search misses.
# But it is not a complete replacement for BM25 -- each approach
# has strengths the other lacks. This is exactly why hybrid search
# (combining both) is so powerful. The next notebook shows how to
# merge results from BM25 and dense retrieval using Reciprocal Rank
# Fusion.

print("Summary:")
print("  Dense retrieval uses neural embeddings to capture meaning")
print("  Cosine similarity measures vector proximity")
print("  ChromaDB stores and retrieves dense vectors efficiently")
print("  Dense search finds semantically related documents")
print("  It complements BM25: together they cover more retrieval scenarios")

Summary:
  Dense retrieval uses neural embeddings to capture meaning
  Cosine similarity measures vector proximity
  ChromaDB stores and retrieves dense vectors efficiently
  Dense search finds semantically related documents
  It complements BM25: together they cover more retrieval scenarios
